# 예제 franka_ex03: FR3 개별 조인트 제어 (self-contained)

Franka FR3(7-DOF) 로봇암의 각 조인트를 하나씩 움직여 어떤 조인트가 어떤 동작을 하는지 시각적으로 확인하는 예제.
기존 6-DOF용 `ex03_joint_goal.ipynb` 를 7-DOF FR3용으로 옮겨온 버전으로, 단일 노트북으로 완결되도록 구성했다.

**6-DOF 예제와 다른 점**
- 조인트가 7개(`fr3_joint1`~`fr3_joint7`) → 위치만 정해지면 자세에 자유도 1만큼 여유가 있다 (redundancy)
- planning group 이름: `fr3_arm`
- base/tip 프레임: `fr3_link0` / `fr3_hand_tcp`
- SRDF에 `home`(올-제로) 자세가 없다 — joint4/joint6 의 한계 때문에 0으로 못 보낸다.
  대신 `ready` 자세를 *기준 자세* 로 쓰고, 각 조인트는 ready 값에서 **delta** 만큼 움직였다 다시 ready 로 복귀한다.
- Gazebo Sim 위에서 도는 환경이라 `use_sim_time=True` 가 필요하다.

**학습 내용**
- 7-DOF 조인트의 redundancy 가 시각적으로 어떻게 드러나는지
- `MoveGroup` 액션 + `MotionPlanRequest` / `JointConstraint` 메시지 구성
- `velocity_scaling_factor` / `acceleration_scaling_factor` 로 속도 조절
- 각 조인트(joint1~joint7)의 역할 직관적 파악

**워크스페이스 크기 참고**
FR3는 reach 약 855mm 의 큰 산업용 로봇이다. 6-DOF 튜토리얼 로봇암(reach ~30cm)보다 동작 반경이 두 배 이상 크므로,
조인트 swing 각도도 더 크게 잡아도 안전하다.

## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz 의 `move_group` 액션 서버에 클라이언트로 붙는 방식이다.
터미널을 둘 띄워야 한다.

> ⚠ 다른 로봇용 MoveIt launch (`demo.launch.xml` 등) 가 떠 있으면 같은 토픽으로 충돌할 수 있다.
> 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — Franka FR3 (Gazebo Sim) + MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch franka_tutorials franka_gazebo_moveit.launch.py
```

Gazebo 가 뜬 뒤 RViz 의 `MotionPlanning` 패널이 보이면 준비 완료.

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab franka_ex03_joint_goal.ipynb
```

셀을 위에서 아래로 순서대로 실행한다 (`Shift+Enter`).

### 다시 실행하고 싶을 때

- `rclpy` 는 한 프로세스에서 한 번만 init 가능하므로, 초기화 셀은 `try/except` 로 감싸 두 번째 실행해도 무시된다.
- 마지막 "정리" 셀까지 실행하지 않고 노트북을 닫아도 된다.

### `use_sim_time` 관련 참고

Gazebo 가 동시에 돌고 있으므로 노드의 시간 기준을 sim time 에 맞춰야 액션 결과가 안정적이다.
이 노트북에서는 노드 생성 시 `Parameter('use_sim_time', value=True)` 를 추가한다.

## 1. 로봇 상수 정의

이 값들은 `franka_description/robots/fr3/fr3.srdf.xacro` 에서 생성되는 SRDF 와 일치한다.
(`hand:=true ee_id:=franka_hand` 옵션이 기본이라 tip link 가 `fr3_hand_tcp` 로 잡힌다.)

In [1]:
PLANNING_GROUP    = 'fr3_arm'
REFERENCE_FRAME   = 'fr3_link0'
END_EFFECTOR_LINK = 'fr3_hand_tcp'
ARM_JOINTS        = ['fr3_joint1', 'fr3_joint2', 'fr3_joint3',
                     'fr3_joint4', 'fr3_joint5', 'fr3_joint6',
                     'fr3_joint7']

## 2. ROS 2 초기화와 노드 생성

이 섹션에서 처음 쓰이는 `rclpy`, `Node`, `ActionClient`, `JointState`, `MoveGroup`, `Parameter` 를 import 한 뒤,
`rclpy.init()` → 노드/액션 클라이언트/구독자 생성 순서로 진행한다.

### 2-1. import (이 섹션에서 처음 쓰이는 것들)

In [2]:
import rclpy
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.parameter import Parameter
from sensor_msgs.msg import JointState
from moveit_msgs.action import MoveGroup

### 2-2. `rclpy` 초기화

한 프로세스에서 한 번만 init 가능하므로, 두 번째 실행은 무시되도록 `try/except` 로 감싼다.

In [3]:
try:
    rclpy.init()
except RuntimeError:
    pass  # 이미 초기화된 경우(노트북에서 재실행)는 무시

### 2-3. 노드 + 액션 클라이언트 + `joint_states` 구독자

Gazebo Sim 과 시간 기준을 맞추기 위해 `use_sim_time=True` 를 노드 파라미터로 넘긴다.

In [4]:
node = Node(
    'franka_ex03_joint_goal_demo',
    parameter_overrides=[Parameter('use_sim_time', value=True)],
)
move_client = ActionClient(node, MoveGroup, 'move_action')

joint_state = {'msg': None}
node.create_subscription(
    JointState, 'joint_states',
    lambda msg: joint_state.update(msg=msg), 10,
)
node.get_logger().info('=== franka_ex03 노트북 노드 생성 완료 ===')

[INFO] [1778392929.423274156] [franka_ex03_joint_goal_demo]: === franka_ex03 노트북 노드 생성 완료 ===


True

## 3. 액션 서버와 `/joint_states` 준비 대기

현재 관절 위치를 한 번 이상 받아본 뒤에 모션 플래닝 요청을 보내야 안정적이다.
이 섹션에서 처음 쓰이는 `time` 모듈을 함께 import 한다.

In [5]:
import time

def wait_for_ready(timeout_sec: float = 30.0) -> None:
    if not move_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('MoveGroup 액션 서버 연결 실패')
    start = time.time()
    while joint_state['msg'] is None:
        rclpy.spin_once(node, timeout_sec=0.1)
        if time.time() - start > timeout_sec:
            raise RuntimeError('joint_states 수신 실패')
    node.get_logger().info('action server + /joint_states 준비됨')

wait_for_ready()

[INFO] [1778392931.390143380] [franka_ex03_joint_goal_demo]: action server + /joint_states 준비됨


## 4. SRDF 에서 `ready` 자세 읽어오기

`move_group` 노드의 `robot_description_semantic` 파라미터에서 SRDF XML 을 받아
`<group_state name="ready" group="fr3_arm">` 안의 조인트 값들을 딕셔너리로 추출한다.

| 함수 | 역할 |
|---|---|
| `fetch_srdf_xml` | `move_group` 에서 SRDF 문자열 받아옴 |
| `parse_named_pose` | SRDF XML 에서 group/name 의 group_state 를 dict 로 파싱 |
| `load_named_pose` | 위 둘을 묶는 진입점 |

### 4-1. `move_group` 에서 SRDF XML 가져오기

In [6]:
from rclpy.parameter_client import AsyncParameterClient

def fetch_srdf_xml(timeout_sec: float = 10.0) -> str:
    client = AsyncParameterClient(node, 'move_group')
    if not client.wait_for_services(timeout_sec=timeout_sec):
        raise RuntimeError('move_group 파라미터 서비스 연결 실패')
    future = client.get_parameters(['robot_description_semantic'])
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)
    return future.result().values[0].string_value

### 4-2. SRDF XML 에서 group_state 파싱

In [7]:
import xml.etree.ElementTree as ET

def parse_named_pose(srdf_xml: str, name: str, group: str) -> dict:
    root = ET.fromstring(srdf_xml)
    for gs in root.findall('group_state'):
        if gs.attrib.get('group') == group and gs.attrib.get('name') == name:
            return {j.attrib['name']: float(j.attrib.get('value', '0'))
                    for j in gs.findall('joint')}
    raise RuntimeError(f'SRDF group_state "{name}" (group={group}) 없음')

### 4-3. 두 단계를 묶는 진입점

In [8]:
def load_named_pose(name: str = 'ready', timeout_sec: float = 10.0) -> dict:
    srdf_xml = fetch_srdf_xml(timeout_sec)
    return parse_named_pose(srdf_xml, name, PLANNING_GROUP)

### 4-4. `ready` 자세 가져와 변수에 저장

FR3 의 `ready` 는 `j2 = -π/4`, `j4 = -3π/4`, `j6 = π/2`, `j7 = π/4` 로 살짝 굽혀 세운 자세다.
j4 / j6 은 한계상 0 으로 못 가므로, 모든 단일 조인트 동작의 *시작/복귀* 자세로 이걸 쓴다.

In [9]:
ready_target = load_named_pose('ready')
node.get_logger().info(f'ready target: {ready_target}')

[INFO] [1778392938.426825541] [franka_ex03_joint_goal_demo]: ready target: {'fr3_joint1': 0.0, 'fr3_joint2': -0.7853981633974483, 'fr3_joint3': 0.0, 'fr3_joint4': -2.356194490192345, 'fr3_joint5': 0.0, 'fr3_joint6': 1.5707963267948966, 'fr3_joint7': 0.7853981633974483}


True

## 5. 코드 한 겹씩 벗기기 — 결론부터

이 섹션이 이 예제의 핵심. 한 조인트를 흔드는 일은 결국 `step_joint('fr3_joint3', ...)` 한 줄로 끝나지만,
그 한 줄이 어떻게 굴러가는지 **바깥에서 안쪽으로** 한 겹씩 벗겨본다.

| 깊이 | 함수 | 역할 |
|---|---|---|
| (1) 사용자 호출 | `step_joint` | 한 조인트만 ready 기준으로 흔들었다 복귀, 회전축 마커도 같이 |
| (2) 한 동작 | `go_to_joint_goal` | 조인트 dict 를 plan request 로 만들어 액션에 던짐 |
| (3) 액션 송수신 | `send_goal_and_wait` | **핵심: `move_client.send_goal_async(goal)`** + accept/result 대기 |
| (4) 데이터 타입 | `make_goal` / `make_plan_request` / `make_joint_constraints` | `goal` 이 받아야 할 메시지 타입을 채워 만드는 작업 |

위에서 아래로 내려가며, 바깥 함수가 안쪽 함수에게 *무엇을* 넘겨줘야 하는지가 점점 또렷해진다.

### 5-1. 가장 바깥 — `step_joint`

사용자가 직접 부르는 함수. 한 조인트만 ready 기준에서 `delta` 만큼 흔들었다가 다시 ready 로 복귀한다.
회전축을 RViz 에 뿌렸다 지우는 일도 같이 한다.

핵심 한 줄은 `go_to_joint_goal(target, ...)` — **조인트 dict 한 덩어리** 만 넘겨주면 된다.
ready 기준 delta 를 쓰는 이유: FR3 의 j4(-3.07~-0.12), j6(0.55~4.52) 한계가 비대칭이라
절대 각도 0 으로 못 가기 때문에, `ready` 값에 delta 를 더하는 방식으로 안전 영역에 머물게 한다.

(여기서 처음 쓰이는 `math` 모듈을 함께 import.)

In [10]:
import math

def step_joint(joint_name: str, delta: float, desc: str = '') -> None:
    deg = math.degrees(delta)
    sign = '+' if delta >= 0 else ''
    node.get_logger().info(f'--- {joint_name} ({desc}) ready 에서 {sign}{deg:.0f}° ---')
    publish_axis_marker(joint_name)
    target = dict(ready_target)
    target[joint_name] = ready_target[joint_name] + delta
    go_to_joint_goal(target, vel=0.3)
    time.sleep(1.5)
    node.get_logger().info('  → ready 복귀')
    go_to_joint_goal(ready_target, vel=0.3)
    time.sleep(1.0)
    delete_axis_marker()

### 5-2. 한 단계 안 — `go_to_joint_goal`

`step_joint` 가 부르는 함수. **조인트 dict 한 덩어리 → 액션 한 번 송신** 의 다리 역할.

내부에서 두 가지를 한다:
1. dict 를 `MotionPlanRequest` 로 변환하고 다시 `MoveGroup.Goal` 로 감싼다 (= req 만들기)
2. `send_goal_and_wait` 로 그 goal 을 액션 서버에 던지고 결과 코드를 받는다

리턴은 success 여부. 구체적으로 *어떻게* 던지는지, *어떤 타입* 이 필요한지는 더 안쪽에서 본다.

In [11]:
def go_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(joint_values, vel, acc)
    code_val = send_goal_and_wait(make_goal(req))
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'MoveGroup 실패 error_code={code_val}')
    return ok

### 5-3. 더 안쪽 — `send_goal_and_wait` 와 **핵심** `send_goal_async`

이 함수가 액션 클라이언트의 본체다. 다섯 줄이지만 이 한 줄이 모든 일의 시작:

```python
send_future = move_client.send_goal_async(goal)   # ← 핵심
```

ROS 2 액션은 비동기라 future 를 두 번 기다린다:

1. `send_goal_async(goal)` → goal 송신, accept 여부를 알려줄 future 받음
2. `spin_until_future_complete` → 그 future 가 끝날 때까지 노드 spin
3. accept 됐으면 `handle.get_result_async()` → 결과 future 받음
4. 다시 `spin_until_future_complete` → 결과 도착할 때까지 spin
5. 결과의 `error_code.val` (정수) 반환 (성공 = `MoveItErrorCodes.SUCCESS` = 1)

여기서 자연스럽게 다음 질문이 따라온다 —
**`send_goal_async` 에 넘기는 `goal` 은 도대체 어떤 데이터 타입이어야 할까?**

In [12]:
from moveit_msgs.msg import MoveItErrorCodes

def send_goal_and_wait(goal: MoveGroup.Goal) -> int:
    send_future = move_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, send_future)
    handle = send_future.result()
    if handle is None or not handle.accepted:
        return MoveItErrorCodes.PLANNING_FAILED
    result_future = handle.get_result_async()
    rclpy.spin_until_future_complete(node, result_future)
    return result_future.result().result.error_code.val

### 5-4. 그래서 `goal` 은 어떤 데이터 타입인가?

`send_goal_async(goal)` 의 `goal` 은 `MoveGroup.Goal` 인스턴스다. 그런데 그 안에는 또 다른 메시지가
중첩돼 있다. 우리가 가지고 있는 건 그냥 파이썬 dict (`{'fr3_joint1': 0.0, ...}`) 인데,
이걸 아래 트리의 모든 필드에 정확히 채워줘야 한다.

```
MoveGroup.Goal
└─ request : MotionPlanRequest
   ├─ group_name = 'fr3_arm'
   ├─ max_velocity_scaling_factor = 0.3
   ├─ max_acceleration_scaling_factor = 0.3
   └─ goal_constraints : [Constraints]
      └─ joint_constraints : [JointConstraint, JointConstraint, ...]
         └─ {joint_name, position, tolerance_above/below, weight}
```

이 트리를 **가장 안쪽부터** 한 겹씩 만들어 올라간다 — `make_joint_constraints` → `make_plan_request` → `make_goal`.

#### 5-4-1. 가장 안쪽 — `JointConstraint` 의 묶음 `Constraints`

조인트 dict 의 각 항목을 `JointConstraint` 한 개로 변환하고, 전부 모아 `Constraints` 한 덩어리로 만든다.

- `tolerance_above` / `tolerance_below`: 목표값에서 ±얼마까지 허용할지 (rad)
- `weight`: 다중 제약일 때 어느 쪽을 더 중시할지 — 여기선 모두 1.0

In [13]:
from moveit_msgs.msg import Constraints, JointConstraint

def make_joint_constraints(joint_values: dict, tol: float = 0.01) -> Constraints:
    constraints = Constraints()
    for jname, val in joint_values.items():
        jc = JointConstraint(joint_name=jname, position=val,
                              tolerance_above=tol, tolerance_below=tol,
                              weight=1.0)
        constraints.joint_constraints.append(jc)
    return constraints

#### 5-4-2. 그 위 — `MotionPlanRequest` 에 담기

플랜 리퀘스트 본체. 위에서 만든 `Constraints` 를 `goal_constraints` 리스트에 추가하고,
플래닝 그룹 이름과 속도/가속 스케일도 같이 채운다.

In [14]:
from moveit_msgs.msg import MotionPlanRequest

def make_plan_request(joints: dict, vel: float, acc: float) -> MotionPlanRequest:
    req = MotionPlanRequest()
    req.group_name = PLANNING_GROUP
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    req.goal_constraints.append(make_joint_constraints(joints))
    return req

#### 5-4-3. 마지막 껍데기 — `MoveGroup.Goal`

액션의 goal 메시지로 한 번 더 감싼다. 이 객체가 바로 `send_goal_async(goal)` 의 인자가 된다.

In [15]:
def make_goal(req: MotionPlanRequest) -> MoveGroup.Goal:
    goal = MoveGroup.Goal()
    goal.request = req
    return goal

## 6. 시나리오 — 먼저 `ready` 자세로

In [16]:
node.get_logger().info('--- ready 자세로 초기 이동 ---')
go_to_joint_goal(ready_target)
time.sleep(1.0)

[INFO] [1778392951.335008676] [franka_ex03_joint_goal_demo]: --- ready 자세로 초기 이동 ---


## 6.5. 회전축 시각화 — RGB 화살표 마커

각 `step_joint` 호출 시, 지령된 조인트의 **회전축**을 RViz 에 화살표로 표시한다.
Franka FR3 의 모든 revolute 조인트는 URDF 상 `axis=(0,0,1)` 이므로,
각 조인트의 *child link* (예: `fr3_joint3` → `fr3_link3`) 의 **+z 축**이 곧 회전축이다.

**색상 규칙** — 그 회전축을 base 프레임(`fr3_link0`) 에서 봤을 때 가장 가까운 월드 축을 따라:

| 월드 축 가장 가까움 | 색상 |
|---|---|
| X | 🔴 빨강 |
| Y | 🟢 초록 |
| Z | 🔵 파랑 |

**RViz 설정** — `MarkerArray` Display 를 추가하고 Topic 을 `/joint_axis_markers` 로 설정.
Fixed Frame 은 `fr3_link0` 그대로.

이 섹션에서 처음 쓰이는 `tf2_ros`, `tf_transformations`, `Marker`, `MarkerArray`, `Point`, `Vector3`, `ColorRGBA` 가 함께 import 된다.

In [17]:
import tf2_ros
import tf_transformations
from visualization_msgs.msg import Marker, MarkerArray
from geometry_msgs.msg import Vector3, Point
from std_msgs.msg import ColorRGBA

AXIS_MARKER_TOPIC = '/joint_axis_markers'
axis_marker_pub = node.create_publisher(MarkerArray, AXIS_MARKER_TOPIC, 10)

tf_buffer = tf2_ros.Buffer()
tf_listener = tf2_ros.TransformListener(tf_buffer, node)

# TF 버퍼가 채워질 때까지 잠깐 spin (joint 별 transform 이 들어와야 함)
for _ in range(30):
    rclpy.spin_once(node, timeout_sec=0.1)
node.get_logger().info('axis_marker_pub + TF listener 준비됨')

[INFO] [1778392954.432073259] [franka_ex03_joint_goal_demo]: axis_marker_pub + TF listener 준비됨


True

### 회전축 마커 헬퍼

| 함수 | 역할 |
|---|---|
| `_world_axis_color(joint_name)` | 그 조인트의 회전축(child link 의 +z)이 base 프레임에서 어느 월드축에 가장 가까운지 보고 색상 반환 |
| `publish_axis_marker(joint_name)` | child link 프레임에서 +z 방향으로 화살표 마커 발행 (조인트 회전과 함께 RViz 에서 자연스럽게 회전) |
| `delete_axis_marker()` | 마커 제거 |

In [18]:
def _joint_child_link(joint_name: str) -> str:
    """fr3_joint3 → fr3_link3 (URDF 상 child link)."""
    return joint_name.replace('joint', 'link')

def _world_axis_color(joint_name: str) -> ColorRGBA:
    """child link 의 +z 축을 base(fr3_link0) 프레임에서 보고
    가장 큰 절대값 성분이 X면 R, Y면 G, Z면 B 로 색상 반환."""
    child = _joint_child_link(joint_name)
    try:
        tf = tf_buffer.lookup_transform(
            REFERENCE_FRAME, child, rclpy.time.Time(),
            timeout=rclpy.duration.Duration(seconds=2.0),
        )
    except Exception as e:
        node.get_logger().warn(f'TF lookup 실패 ({child}): {e} → 회색으로 대체')
        return ColorRGBA(r=0.5, g=0.5, b=0.5, a=0.9)
    q = tf.transform.rotation
    M = tf_transformations.quaternion_matrix([q.x, q.y, q.z, q.w])
    z_world = M[:3, 2]   # child link 의 +z 가 월드에서 향하는 방향
    abs_v = [abs(z_world[0]), abs(z_world[1]), abs(z_world[2])]
    i = abs_v.index(max(abs_v))
    if i == 0:
        return ColorRGBA(r=1.0, g=0.0, b=0.0, a=0.9)   # X → 빨강
    if i == 1:
        return ColorRGBA(r=0.0, g=1.0, b=0.0, a=0.9)   # Y → 초록
    return ColorRGBA(r=0.0, g=0.0, b=1.0, a=0.9)       # Z → 파랑

def publish_axis_marker(joint_name: str, length: float = 0.40) -> None:
    """회전축을 child link 프레임에 +z 방향 화살표로 표시.

    arrow points 는 child 의 -z/2 → +z/2 (조인트 pivot 중심을 통과하는 길이 length 화살표).
    """
    child = _joint_child_link(joint_name)
    color = _world_axis_color(joint_name)
    m = Marker()
    m.header.frame_id = child
    m.header.stamp = node.get_clock().now().to_msg()
    m.ns = 'joint_axis'
    m.id = 0
    m.type = Marker.ARROW
    m.action = Marker.ADD
    m.points = [Point(x=0.0, y=0.0, z=-length / 2),
                Point(x=0.0, y=0.0, z=+length / 2)]
    m.scale = Vector3(x=0.015, y=0.030, z=0.030)   # shaft dia, head dia, head len
    m.color = color
    axis_marker_pub.publish(MarkerArray(markers=[m]))
    node.get_logger().info(
        f'  ↻ {joint_name} 회전축 표시 (child={child}, color RGB={color.r:.0f},{color.g:.0f},{color.b:.0f})'
    )

def delete_axis_marker() -> None:
    m = Marker()
    m.header.frame_id = REFERENCE_FRAME
    m.ns = 'joint_axis'
    m.id = 0
    m.action = Marker.DELETE
    axis_marker_pub.publish(MarkerArray(markers=[m]))

## 7. 각 조인트 개별 이동 — `step_joint` 호출

이제 5-1 에서 정의한 `step_joint(joint_name, delta, desc)` 를 조인트별로 한 번씩 호출한다.
각 호출 = `joint_name` 만 `ready_target[joint] + delta` 로 보낸 뒤 다시 ready 로 복귀.

(다시 강조: FR3 의 j4(-3.07~-0.12), j6(0.55~4.52) 한계가 비대칭이라 절대 각도 0 으로 못 보낸다.
그래서 `ready` 기준 delta 방식을 쓴다.)

### fr3_joint1 — 베이스 회전 (Yaw)

In [19]:
step_joint('fr3_joint1', math.radians(60), '베이스 회전')

[INFO] [1778392974.622591084] [franka_ex03_joint_goal_demo]: --- fr3_joint1 (베이스 회전) ready 에서 +60° ---
[INFO] [1778392974.624808863] [franka_ex03_joint_goal_demo]:   ↻ fr3_joint1 회전축 표시 (child=fr3_link1, color RGB=0,0,1)
[INFO] [1778392978.204400782] [franka_ex03_joint_goal_demo]:   → ready 복귀


### fr3_joint2 — 어깨 (Pitch, 앞뒤 기울기)

In [20]:
step_joint('fr3_joint2', math.radians(-30), '어깨 앞뒤 기울기')

[INFO] [1778392993.961250073] [franka_ex03_joint_goal_demo]: --- fr3_joint2 (어깨 앞뒤 기울기) ready 에서 -30° ---
[INFO] [1778392993.963066323] [franka_ex03_joint_goal_demo]:   ↻ fr3_joint2 회전축 표시 (child=fr3_link2, color RGB=0,1,0)
[INFO] [1778392997.443370361] [franka_ex03_joint_goal_demo]:   → ready 복귀


### fr3_joint3 — 어깨 비틀기 (Roll)

In [21]:
step_joint('fr3_joint3', math.radians(45), '어깨 비틀기')

[INFO] [1778393000.678500531] [franka_ex03_joint_goal_demo]: --- fr3_joint3 (어깨 비틀기) ready 에서 +45° ---
[INFO] [1778393000.680142790] [franka_ex03_joint_goal_demo]:   ↻ fr3_joint3 회전축 표시 (child=fr3_link3, color RGB=0,0,1)
[INFO] [1778393004.259487250] [franka_ex03_joint_goal_demo]:   → ready 복귀


### fr3_joint4 — 팔꿈치 (Pitch, 굽힘)

*ready 의 j4 = -3π/4 ≈ -135°. 0 방향으로만 움직일 수 있다.*

In [22]:
step_joint('fr3_joint4', math.radians(45), '팔꿈치 굽힘')

[INFO] [1778393007.860287229] [franka_ex03_joint_goal_demo]: --- fr3_joint4 (팔꿈치 굽힘) ready 에서 +45° ---
[INFO] [1778393007.862041965] [franka_ex03_joint_goal_demo]:   ↻ fr3_joint4 회전축 표시 (child=fr3_link4, color RGB=0,1,0)
[INFO] [1778393011.240689840] [franka_ex03_joint_goal_demo]:   → ready 복귀


### fr3_joint5 — 손목 비틀기 (Roll)

In [23]:
step_joint('fr3_joint5', math.radians(45), '손목 비틀기')

[INFO] [1778393016.005543653] [franka_ex03_joint_goal_demo]: --- fr3_joint5 (손목 비틀기) ready 에서 +45° ---
[INFO] [1778393016.007296326] [franka_ex03_joint_goal_demo]:   ↻ fr3_joint5 회전축 표시 (child=fr3_link5, color RGB=1,0,0)
[INFO] [1778393019.235990668] [franka_ex03_joint_goal_demo]:   → ready 복귀


### fr3_joint6 — 손목 (Pitch, 굽힘)

*ready 의 j6 = π/2 ≈ 90°. 한계는 0.55~4.52 이므로 좌우 모두 가능.*

In [24]:
step_joint('fr3_joint6', math.radians(30), '손목 굽힘')

[INFO] [1778393023.003937848] [franka_ex03_joint_goal_demo]: --- fr3_joint6 (손목 굽힘) ready 에서 +30° ---
[INFO] [1778393023.006033994] [franka_ex03_joint_goal_demo]:   ↻ fr3_joint6 회전축 표시 (child=fr3_link6, color RGB=0,1,0)
[INFO] [1778393025.734474419] [franka_ex03_joint_goal_demo]:   → ready 복귀


### fr3_joint7 — 그리퍼 회전 (Yaw)

In [25]:
step_joint('fr3_joint7', math.radians(45), '그리퍼 회전')

[INFO] [1778393028.754606969] [franka_ex03_joint_goal_demo]: --- fr3_joint7 (그리퍼 회전) ready 에서 +45° ---
[INFO] [1778393028.756334304] [franka_ex03_joint_goal_demo]:   ↻ fr3_joint7 회전축 표시 (child=fr3_link7, color RGB=0,0,1)
[INFO] [1778393031.785521699] [franka_ex03_joint_goal_demo]:   → ready 복귀


## 8. 보너스 — 여러 조인트 동시 이동

각 조인트가 서로 다른 목표값을 가질 때 MoveIt 이 7-DOF 공간에서 한 번에 보간해 보낸다.
여기서도 `ready` 기준 delta 로 정의해 안전한 영역에 머물게 한다.

In [26]:
multi_target = {jname: ready_target[jname] + d for jname, d in {
    'fr3_joint1': math.radians( 30),
    'fr3_joint2': math.radians(-15),
    'fr3_joint3': math.radians( 30),
    'fr3_joint4': math.radians( 30),
    'fr3_joint5': math.radians( 30),
    'fr3_joint6': math.radians( 20),
    'fr3_joint7': math.radians(-30),
}.items()}

# 7 개 조인트가 동시에 회전 → 7 개 회전축을 한 번에 표시
for i, jname in enumerate(ARM_JOINTS):
    child = _joint_child_link(jname)
    color = _world_axis_color(jname)
    m = Marker()
    m.header.frame_id = child
    m.header.stamp = node.get_clock().now().to_msg()
    m.ns = 'joint_axis'
    m.id = i
    m.type = Marker.ARROW
    m.action = Marker.ADD
    m.points = [Point(x=0.0, y=0.0, z=-0.10), Point(x=0.0, y=0.0, z=0.10)]
    m.scale = Vector3(x=0.015, y=0.030, z=0.030)
    m.color = color
    axis_marker_pub.publish(MarkerArray(markers=[m]))

node.get_logger().info('--- 여러 조인트 동시 이동 ---')
go_to_joint_goal(multi_target)
time.sleep(2.0)
go_to_joint_goal(ready_target)

# 7 개 마커 모두 삭제
for i in range(len(ARM_JOINTS)):
    m = Marker()
    m.header.frame_id = REFERENCE_FRAME
    m.ns = 'joint_axis'
    m.id = i
    m.action = Marker.DELETE
    axis_marker_pub.publish(MarkerArray(markers=[m]))

node.get_logger().info('=== franka_ex03 완료! ===')

[INFO] [1778393034.325152420] [franka_ex03_joint_goal_demo]: --- 여러 조인트 동시 이동 ---
[INFO] [1778393039.732384651] [franka_ex03_joint_goal_demo]: === franka_ex03 완료! ===


True

## 9. 정리

노트북을 닫기 전에 노드와 rclpy 를 안전하게 정리한다.
다시 실행하려면 위쪽 셀부터 순서대로 다시 돌리면 된다.

In [28]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass